# TP5 — Foundation models for robotics: action heads, action chunks and language
**DLA · Deep learning · TP5** · Tuesday 17 November 2026 · 13:45–15:45 · Esclangon 221
Silvia Tulli — Sorbonne Université, ISIR

This practical session follows lecture **C5 “Foundation models for robotics”**. In the lecture we saw that modern vision-language-action models (VLAs) such as π0, SmolVLA and GR00T share three design choices:

1. a **generative action head** (diffusion / flow matching) instead of plain regression,
2. **action chunking**: predicting several future actions at once,
3. **language conditioning**: the instruction is an input to the policy.

Today you will rebuild each of these ideas **from scratch, on small 2D problems that train in seconds**, so you can see *why* they matter. In the last part you will load a real open VLA (SmolVLA) and a real robot dataset with 🤗 LeRobot.

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | Regression vs flow matching on multimodal demonstrations | 35 min |
| 2 | Closed loop and action chunking | 30 min |
| 3 | Language-conditioned policy and grounding tests | 30 min |
| 4 | A real VLA: SmolVLA and a LeRobot dataset (GPU + internet) | 15 min |
| 5 | Wrap-up questions | 5 min |

**How to work.** Cells marked `# TODO` are for you to complete. Questions marked **Q** are to answer in a markdown cell (short answers are fine). Run the notebook top to bottom. Parts 1–3 run on CPU; Part 4 needs a GPU runtime (*Runtime → Change runtime type → T4 GPU* on Colab).

## Part 0 — Setup

In [ ]:
import math, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

def seed_all(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## Part 1 — Many demonstrations, more than one right answer (35 min)

A robot starts below an obstacle and must reach a goal above it. Half of the human demonstrators go around the obstacle **on the left**, half **on the right**. Both are correct.

We first look at a single decision: from the start position $s$, predict the **waypoint** $a\in\mathbb{R}^2$ the robot should go to next (left of the obstacle or right of it).

In [ ]:
OBST = dict(x0=-0.45, x1=0.45, y0=-0.35, y1=0.35)        # the obstacle (a box)
LEFT, RIGHT = np.array([-1.0, 0.0]), np.array([1.0, 0.0])  # the two valid waypoints

def in_obstacle(p):
    p = np.atleast_2d(p)
    return (p[:, 0] > OBST["x0"]) & (p[:, 0] < OBST["x1"]) & (p[:, 1] > OBST["y0"]) & (p[:, 1] < OBST["y1"])

def make_waypoint_data(n=2000, noise=0.08):
    s = np.stack([np.random.uniform(-0.3, 0.3, n), np.random.uniform(-1.6, -1.2, n)], 1)   # start positions
    side = np.random.rand(n) < 0.5
    a = np.where(side[:, None], LEFT, RIGHT) + noise * np.random.randn(n, 2)               # chosen waypoint
    return s.astype(np.float32), a.astype(np.float32)

S1, A1 = make_waypoint_data()

def draw_scene(ax, title=""):
    ax.add_patch(plt.Rectangle((OBST["x0"], OBST["y0"]), OBST["x1"]-OBST["x0"], OBST["y1"]-OBST["y0"], color="#bbbbbb"))
    ax.plot(0, 1.5, "*", ms=18, color="gold", mec="k"); ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.8, 1.8)
    ax.set_aspect("equal"); ax.set_title(title)

fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_scene(ax, "Demonstrations: start (grey) → waypoint")
ax.scatter(S1[:300, 0], S1[:300, 1], s=6, c="grey"); ax.scatter(A1[:300, 0], A1[:300, 1], s=6, c="tab:blue"); plt.show()

### 1.1 Behaviour cloning with mean-squared-error regression
We train a small MLP $\pi_\theta(s)$ with the usual supervised loss $\mathcal{L}=\lVert \pi_\theta(s)-a\rVert^2$.

In [ ]:
def mlp(i, o, h=128, n=3):
    layers, d = [], i
    for _ in range(n): layers += [nn.Linear(d, h), nn.SiLU()]; d = h
    return nn.Sequential(*layers, nn.Linear(d, o))

def train(model, loss_fn, steps=2000, bs=256, lr=2e-3, data=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr); X = [torch.tensor(d) for d in data]; n = len(X[0])
    for it in range(steps):
        idx = torch.randint(0, n, (bs,)); loss = loss_fn(model, *[x[idx] for x in X])
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 500 == 0 or it == steps - 1: print(f"step {it:5d}  loss {loss.item():.4f}")
    return model

# TODO 1.1 — write the MSE loss between the model prediction and the demonstrated waypoint
def mse_loss(model, s, a):
    # --- your code here ---
    raise NotImplementedError("Complete this TODO")

seed_all(0)
reg = train(mlp(2, 2), mse_loss, data=(S1, A1))

In [ ]:
with torch.no_grad():
    s_test = torch.tensor(make_waypoint_data(300)[0]); pred = reg(s_test).numpy()
fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_scene(ax, "MSE regression: predicted waypoints")
ax.scatter(pred[:, 0], pred[:, 1], s=8, c="tab:red"); plt.show()
print(f"Predicted waypoints inside the obstacle: {in_obstacle(pred).mean():.0%}")

**Q1.** Where do the regression predictions land, and why? Relate your answer to the minimiser of the expected squared error.

### 1.2 Flow matching: learn to transport noise into actions
Flow matching (used by π0 and SmolVLA) learns a **velocity field** $v_\theta(x_t, t, s)$. For a demonstrated action $a$ and a noise sample $x_0\sim\mathcal{N}(0, I)$, define the straight path
$$x_t = (1-t)\,x_0 + t\,a, \qquad t\sim\mathcal{U}[0,1].$$
Its velocity is constant, $\dot x_t = a - x_0$, so the training loss is
$$\mathcal{L}_{\text{FM}} = \mathbb{E}\,\lVert v_\theta(x_t, t, s) - (a - x_0)\rVert^2 .$$
At test time, start from noise and integrate $\dot x = v_\theta(x, t, s)$ from $t=0$ to $t=1$ (Euler steps).

In [ ]:
class FlowPolicy(nn.Module):
    # velocity network v(x_t, t, cond); act_dim can be a whole action chunk (flattened)
    def __init__(self, cond_dim, act_dim, h=256):
        super().__init__(); self.act_dim = act_dim
        self.net = mlp(act_dim + 1 + cond_dim, act_dim, h=h, n=3)
    def forward(self, x_t, t, cond):
        return self.net(torch.cat([x_t, t, cond], -1))

# TODO 1.2a — flow-matching loss
def fm_loss(model, cond, a):
    # --- your code here ---
    raise NotImplementedError("Complete this TODO")

# TODO 1.2b — sampling with K Euler steps
@torch.no_grad()
def fm_sample(model, cond, K=20):
    # --- your code here ---
    raise NotImplementedError("Complete this TODO")

seed_all(0)
flow = train(FlowPolicy(2, 2), fm_loss, steps=3000, data=(S1, A1))

In [ ]:
samp = fm_sample(flow, s_test).numpy()
fig, axs = plt.subplots(1, 2, figsize=(9, 4.5))
draw_scene(axs[0], "MSE regression"); axs[0].scatter(pred[:, 0], pred[:, 1], s=8, c="tab:red")
draw_scene(axs[1], "Flow matching samples"); axs[1].scatter(samp[:, 0], samp[:, 1], s=8, c="tab:blue")
plt.show()
print(f"inside obstacle — regression: {in_obstacle(pred).mean():.0%} | flow matching: {in_obstacle(samp).mean():.0%}")
print(f"flow samples going left: {(samp[:,0] < 0).mean():.0%}")

**Q2.** Visualise the transport: plot the samples after 0, 5, 10 and 20 Euler steps (re-use `fm_sample` with a modified loop). How many steps do you need before the two modes are clearly separated? What does this cost at inference time on a robot running at 50 Hz?

In [ ]:
# TODO (Q2) — plot intermediate samples of the flow at several integration steps
# --- your code here ---
raise NotImplementedError("Complete this TODO")

### 1.3 (bonus) Actions as discrete tokens, RT-2 style
RT-2 and OpenVLA write each action dimension as one of 256 bins and predict it with a softmax, like a word. Implement a classifier with `NB` bins per dimension (independent softmax per dimension), train it with cross-entropy and **sample** from it.

**Q3.** Does the token head capture both modes? What is the precision limit with `NB` bins on $[-2, 2]$? What could go wrong for a task where the *x* and *y* choices are correlated (hint: the heads are independent)?

In [ ]:
NB, LO, HI = 64, -2.0, 2.0
def to_bins(a):   return ((a - LO) / (HI - LO) * NB).long().clamp(0, NB - 1)
def from_bins(b): return LO + (b.float() + 0.5) * (HI - LO) / NB

# TODO 1.3 — token head: model outputs 2*NB logits; loss = cross-entropy for each dimension
tok = mlp(2, 2 * NB)
def token_loss(model, s, a):
    # --- your code here ---
    raise NotImplementedError("Complete this TODO")
seed_all(0); train(tok, token_loss, steps=2000, data=(S1, A1))
with torch.no_grad():
    probs = tok(s_test).view(-1, 2, NB).softmax(-1)
    samp_tok = from_bins(torch.distributions.Categorical(probs).sample()).numpy()
fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_scene(ax, "Token head samples"); ax.scatter(samp_tok[:, 0], samp_tok[:, 1], s=8, c="tab:green"); plt.show()

## Part 2 — Closed loop and action chunking (30 min)

A policy does not take one decision: it acts **in a loop**, and each action changes the next observation. Now the demonstrations are full trajectories of $T=40$ steps around the obstacle (left or right). The action is the displacement for the next step.

**Partial observation.** The robot's sensor only measures how far it is from the goal (its height $y$) and the elapsed time, **not its lateral position $x$** (think of a camera that cannot see the arm, or an occluded scene). This is common on real robots: the observation does not tell the policy which way it had already decided to go.

We compare two policies trained with flow matching:

- **H = 1**: predict one action, execute it, observe, predict again (a new noise sample every step);
- **H = 8**: predict a **chunk** of 8 future actions, execute them all, then predict again (as in ACT, π0, SmolVLA).

In [ ]:
T = 40
def demo_trajectory(side):
    # smooth path from start to goal that goes around the obstacle on one side
    start, goal = np.array([np.random.uniform(-0.15, 0.15), -1.5]), np.array([0.0, 1.5])
    u = np.linspace(0, 1, T + 1)
    x = side * 1.0 * np.sin(np.pi * u) + start[0] * (1 - u)
    y = start[1] + (goal[1] - start[1]) * u
    return np.stack([x, y], 1) + 0.01 * np.random.randn(T + 1, 2)

def make_chunk_data(H, n_demos=400):
    S, A = [], []
    for _ in range(n_demos):
        p = demo_trajectory(np.random.choice([-1, 1])); d = np.diff(p, axis=0)            # T displacements
        d = np.concatenate([d, np.repeat(d[-1:], H, 0)])                                   # pad the end
        for t in range(T):
            S.append(np.array([p[t][1], t / T])); A.append(d[t:t + H].reshape(-1))   # observation = (y, progress): x is NOT observed
    return np.array(S, np.float32), np.array(A, np.float32)

fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_scene(ax, "Trajectory demonstrations")
for _ in range(20): p = demo_trajectory(np.random.choice([-1, 1])); ax.plot(p[:, 0], p[:, 1], lw=1, alpha=.6)
plt.show()

In [ ]:
SCALE = 20.0   # displacements are small: scale them up for training
def train_chunk_policy(H, steps=6000):
    S, A = make_chunk_data(H); seed_all(1)
    pol = FlowPolicy(2, 2 * H)
    return train(pol, fm_loss, steps=steps, data=(S, A * SCALE))

pol_H1 = train_chunk_policy(1)
pol_H8 = train_chunk_policy(8)

### 2.1 Roll out the policies in closed loop
Complete `rollout`: query the policy, execute the first `n_exec` actions of the predicted chunk, re-observe, repeat. Count a **collision** if the robot ever enters the obstacle, and a **success** if it ends near the goal without collision.

In [ ]:
@torch.no_grad()
def rollout(pol, H, n_exec, start=None):
    p = np.array([np.random.uniform(-0.15, 0.15), -1.5]) if start is None else np.array(start, float)
    path, t = [p.copy()], 0
    while t < T:
        cond = torch.tensor([[p[1], t / T]], dtype=torch.float32)   # the policy only sees (y, progress)
        # TODO 2.1 — sample a chunk, reshape it to (H, 2), undo the scaling, execute the first n_exec steps
        # --- your code here ---
        raise NotImplementedError("Complete this TODO")
    path = np.array(path)
    collided = in_obstacle(path).any(); success = (not collided) and np.linalg.norm(path[-1] - [0, 1.5]) < 0.25
    return path, collided, success

def evaluate(pol, H, n_exec, n=100):
    res = [rollout(pol, H, n_exec) for _ in range(n)]
    return res, np.mean([r[1] for r in res]), np.mean([r[2] for r in res])

seed_all(2)
for name, pol, H, ne in [("H=1, re-plan every step", pol_H1, 1, 1), ("H=8, execute 8", pol_H8, 8, 8), ("H=8, execute 1", pol_H8, 8, 1)]:
    res, col, suc = evaluate(pol, H, ne)
    print(f"{name:26s} collisions {col:.0%} · success {suc:.0%}")
    fig, ax = plt.subplots(figsize=(4, 4)); draw_scene(ax, f"{name}\ncollisions {col:.0%} · success {suc:.0%}")
    for r in res[:25]: ax.plot(r[0][:, 0], r[0][:, 1], lw=1, alpha=.7)
    plt.show()

**Q4.** Compare the three settings. Look at the H = 1 trajectories: why do they zig-zag and end up in the obstacle? Why does executing a whole chunk help, and why does *H = 8, execute 1* lose most of the benefit? (Hint: how many *independent* noise samples decide the side in each setting? With H = 8 there are still 5 of them: why is the collision rate not 0?)

**Q4b (optional).** Give the policy its full position (observation $(x, y, t)$, i.e. `FlowPolicy(3, 2*H)`) and re-run. What happens to H = 1, and why? What does this tell you about *when* chunking matters most?

**Q5.** Measure smoothness: compute the mean squared change of displacement (jerk proxy) along the paths for each setting. Which is smoother, and why does this matter on a real arm?

**Q6.** Chunking also has a cost. What happens if the environment changes in the middle of a chunk (e.g. someone moves the obstacle)? How do SmolVLA's *asynchronous inference* and ACT's *temporal ensembling* address this trade-off?

In [ ]:
# TODO (Q5) — jerk proxy for each setting
# --- your code here ---
raise NotImplementedError("Complete this TODO")

## Part 3 — Language-conditioned policy (30 min)

Now the robot sees **three coloured targets** (red, green, blue) placed at random positions, and receives an instruction such as *“go to the red target”*. The state contains the robot position and, for each target, its position and its colour. The order of the targets in the state is shuffled, so the policy **must use the instruction** to pick the right one.

The instruction is encoded by a **frozen pretrained sentence encoder** (as a VLA uses a frozen or fine-tuned language model). If the model cannot be downloaded, the notebook falls back to a bag-of-words encoder; compare the two if you can.

In [ ]:
COLORS = ["red", "green", "blue"]
TRAIN_TEMPLATES = ["go to the {} target", "move to the {} one", "reach the {} goal", "navigate to the {} marker"]
TEST_PARAPHRASES = ["please head over to the {} dot", "drive towards the {} circle", "can you get to the {} spot"]

# ---- instruction encoder: pretrained sentence encoder if available, else bag of words
try:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    def encode(texts): return torch.tensor(_st.encode(list(texts), normalize_embeddings=True), dtype=torch.float32)
    ENCODER = "all-MiniLM-L6-v2 (pretrained)"
except Exception as e:
    VOCAB = sorted({w for t in TRAIN_TEMPLATES + TEST_PARAPHRASES for w in t.replace("{}", "").split()} | set(COLORS))
    def encode(texts):
        out = torch.zeros(len(texts), len(VOCAB))
        for i, t in enumerate(texts):
            for w in t.lower().split():
                if w in VOCAB: out[i, VOCAB.index(w)] = 1.0
        return F.normalize(out, dim=-1)
    ENCODER = f"bag of words (fallback: {type(e).__name__})"
EMB = encode(["go to the red target"]).shape[1]
print("Instruction encoder:", ENCODER, "| embedding size", EMB)

In [ ]:
def make_lang_episode(templates):
    targets = np.random.uniform(-1.3, 1.3, (3, 2)); order = np.random.permutation(3)
    k = np.random.randint(3); goal = targets[k]
    instr = random.choice(templates).format(COLORS[k])
    slots = np.concatenate([np.concatenate([targets[j], np.eye(3)[j]]) for j in order])   # (x, y, one-hot colour) × 3
    return targets, k, instr, slots

def make_lang_data(n=6000, templates=TRAIN_TEMPLATES):
    X, L, A = [], [], []
    for _ in range(n):
        targets, k, instr, slots = make_lang_episode(templates)
        p = np.random.uniform(-1.5, 1.5, 2); d = targets[k] - p
        step = d / max(np.linalg.norm(d), 1e-6) * min(0.15, np.linalg.norm(d))           # expert: straight to the goal
        X.append(np.concatenate([p, slots])); L.append(instr); A.append(step)
    return np.array(X, np.float32), encode(L).numpy(), np.array(A, np.float32)

Xl, El, Al = make_lang_data(12000)
print(Xl.shape, El.shape, Al.shape)

### 3.1 Conditioning the policy on language with FiLM
A simple and widely used way to inject language is **FiLM**: the instruction embedding produces a scale $\gamma$ and a shift $\beta$ that modulate the hidden features of the state network, $h \leftarrow \gamma(\ell)\odot h + \beta(\ell)$. Complete the module. (The expert is unimodal here, so a regression head is enough.)

In [ ]:
class FiLMPolicy(nn.Module):
    def __init__(self, state_dim, emb_dim, h=256):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(state_dim, h), nn.SiLU())
        self.film = nn.Linear(emb_dim, 2 * h)
        self.out = nn.Sequential(nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, e):
        # TODO 3.1 — compute gamma and beta from e, modulate the hidden features, return the action
        # --- your code here ---
        raise NotImplementedError("Complete this TODO")

def lang_loss(model, x, e, a): return F.mse_loss(model(x, e), a)
seed_all(0)
lang_pol = train(FiLMPolicy(Xl.shape[1], EMB), lang_loss, steps=6000, data=(Xl, El, Al))

### 3.2 Evaluate: seen templates, paraphrases, and a counterfactual test
Roll out the policy for 40 steps and count a success if the robot ends within 0.15 of the **instructed** target.

In [ ]:
@torch.no_grad()
def lang_rollout(pol, targets, slots, instr, steps=40):
    p = np.random.uniform(-1.5, 1.5, 2); e = encode([instr]); path = [p.copy()]
    for _ in range(steps):
        x = torch.tensor(np.concatenate([p, slots])[None], dtype=torch.float32)
        p = p + pol(x, e).numpy()[0]; path.append(p.copy())
    return np.array(path)

def lang_success_rate(pol, templates, n=200):
    ok = 0
    for _ in range(n):
        targets, k, instr, slots = make_lang_episode(templates)
        ok += np.linalg.norm(lang_rollout(pol, targets, slots, instr)[-1] - targets[k]) < 0.15
    return ok / n

seed_all(4)
print(f"success, training templates : {lang_success_rate(lang_pol, TRAIN_TEMPLATES):.0%}")
print(f"success, unseen paraphrases : {lang_success_rate(lang_pol, TEST_PARAPHRASES):.0%}")

In [ ]:
# Counterfactual test: same scene, same start, only the colour word changes
seed_all(5)
targets, k, instr, slots = make_lang_episode(TRAIN_TEMPLATES)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for j, c in enumerate(COLORS):
    ax.plot(*targets[j], "o", ms=14, color=c, mec="k")
    np.random.seed(7); path = lang_rollout(lang_pol, targets, slots, f"go to the {c} target")
    ax.plot(path[:, 0], path[:, 1], "-", color=c, lw=2, label=f"“go to the {c} target”")
ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.8, 1.8); ax.set_aspect("equal"); ax.legend(fontsize=8); plt.show()

**Q7.** Does the policy follow the instruction (counterfactual plot)? What happens with unseen paraphrases, and why does it depend on the encoder (pretrained sentence encoder vs bag of words)? Relate this to why VLAs start from a **pretrained** vision-language model.

**Q8.** Try an instruction with a colour that never appears in training (e.g. *“go to the yellow target”*) or a negation (*“do not go to the red target”*). What does the policy do? What does this tell you about grounding?

In [ ]:
# TODO (Q8) — try out-of-distribution instructions and plot the result
# --- your code here ---
raise NotImplementedError("Complete this TODO")

## Part 4 — A real VLA: SmolVLA and a LeRobot dataset (15 min, GPU + internet)

We now look at the real thing: **SmolVLA** (≈450M parameters: a small VLM + a flow-matching action expert, the same recipe as Part 1–2 at scale) and a community dataset recorded with low-cost SO-100 arms, both hosted on the Hugging Face Hub and used through the **LeRobot** library.

⚠️ This part needs a **GPU runtime** and internet access, and downloads about 2 GB. The LeRobot API changes between versions: if a cell fails, read the code and answer the questions anyway.

In [ ]:
# Run once (takes 2–4 minutes on Colab). Restart the runtime if asked.
# !pip install -q "lerobot[smolvla]"

In [ ]:
DATASET = "lerobot/svla_so100_pickplace"   # community SO-100 pick-and-place dataset (fallback: "lerobot/pusht")
try:
    try:
        from lerobot.datasets.lerobot_dataset import LeRobotDataset          # recent versions
    except ImportError:
        from lerobot.common.datasets.lerobot_dataset import LeRobotDataset   # older versions
    ds = LeRobotDataset(DATASET)
    print(ds)
    item = ds[0]
    for k, v in item.items():
        print(f"{k:35s}", tuple(v.shape) if hasattr(v, "shape") else v)
except Exception as e:
    print("Could not load the dataset:", type(e).__name__, e)

**Q9.** Inspect one sample. Which keys are observations (cameras, robot state) and which is the action? What is the action dimension, and what does each component correspond to on the SO-100 arm? Where is the language instruction stored?

In [ ]:
# Plot the actions of one episode over time
try:
    ep = 0
    idx = [i for i in range(len(ds)) if int(ds[i]["episode_index"]) == ep][:300]
    acts = torch.stack([ds[i]["action"] for i in idx]).numpy()
    plt.figure(figsize=(9, 3)); plt.plot(acts); plt.title(f"Episode {ep}: action components over time"); plt.xlabel("frame"); plt.show()
    cam_keys = [k for k in ds[idx[0]].keys() if "image" in k]
    fig, axs = plt.subplots(1, len(cam_keys), figsize=(4 * len(cam_keys), 3))
    for ax, k in zip(np.atleast_1d(axs), cam_keys): ax.imshow(ds[idx[0]][k].permute(1, 2, 0).numpy()); ax.set_title(k); ax.axis("off")
    plt.show()
except Exception as e:
    print("Skipped:", type(e).__name__, e)

In [ ]:
# Load the pretrained SmolVLA base model and predict an action chunk for one frame
try:
    try:
        from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    except ImportError:
        from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base").to(device).eval()
    print(f"SmolVLA parameters: {sum(p.numel() for p in policy.parameters())/1e6:.0f} M")
    batch = {k: (v.unsqueeze(0).to(device) if torch.is_tensor(v) else [v]) for k, v in ds[idx[50]].items()}
    with torch.no_grad():
        a_pred = policy.select_action(batch)
    print("predicted action:", a_pred.cpu().numpy().round(3))
    print("demonstrated    :", ds[idx[50]]["action"].numpy().round(3))
except Exception as e:
    print("SmolVLA step skipped:", type(e).__name__, e)

**Q10.** The base model was pretrained on community data but **not fine-tuned on this dataset**. Is the predicted action close to the demonstrated one? List what you would need to do to deploy SmolVLA on your own SO-100 arm (data, fine-tuning, inference rate, evaluation), using the vocabulary of the lecture.

## Part 5 — Wrap-up (5 min)

Answer in two or three sentences each:

1. Why do π0, SmolVLA and GR00T use flow matching or diffusion action heads rather than MSE regression? Use your Part 1 plots.
2. What two problems does action chunking address, and what new problem does it create?
3. In Part 3, what made the policy generalise (or not) to new phrasings? How does this relate to starting a VLA from a pretrained VLM, and to the risk of *catastrophic forgetting* when fine-tuning it?
4. Name one evaluation you would add before trusting a VLA on a real robot, and say what it would reveal that a single success rate would not.